# LandslideRisk-48: reference solution
Plan: (1) load data, (2) build features from each reading and from *earlier* readings at the
same site, (3) validate with folds grouped by site, because every test site is new, (4) compare
simple rankers against learned models, (5) refit the best model and output each day's top 3 sites.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 48
K = 3
TARGET = "landslide_risk_48h"
DATA = Path("./dataset/public")
OUT = Path("./working")
OUT.mkdir(parents=True, exist_ok=True)

# Readings: one row per site and day. Queries: one row per day with its candidate sites.
# Query IDs are "train_YYYY-MM-DD" in train files and "test_YYYY-MM-DD" in test files.
train = pd.read_csv(DATA / "train.csv", parse_dates=["timestamp"])
test = pd.read_csv(DATA / "test_readings.csv", parse_dates=["timestamp"])
test_queries = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print(train.shape, test.shape, test_queries.shape, sample.shape)
print("train sites:", train.location_id.nunique(), "| test sites:", test.location_id.nunique(),
      "| overlap:", len(set(train.location_id) & set(test.location_id)))
print("candidates per test query:", test_queries.candidate_sites.str.split().str.len().unique())
print("positive rate:", round(train[TARGET].mean(), 4))

## Metric
Same as the grader: MAP@3 over days with at least one landslide among the candidate sites
(test.csv only contains such days).

In [ ]:
def map_at_k(df, score_col, k=K):
    """df has query_id, location_id, TARGET and a score column; higher score = recommend first."""
    total, n = 0.0, 0
    for _, g in df.groupby("query_id"):
        n_rel = int(g[TARGET].sum())
        if n_rel == 0:
            continue
        top = g.sort_values([score_col, "location_id"], ascending=[False, True])[TARGET].to_numpy()[:k]
        hits, ap = 0, 0.0
        for rank, rel in enumerate(top, start=1):
            if rel:
                hits += 1
                ap += hits / rank
        total += ap / min(k, n_rel)
        n += 1
    return total / n

## Features
Each site's rows are processed separately, in timestamp order, and only the current and
earlier readings are used. Train and test sites never mix, so concatenating the two frames
only saves code. `location_id` is not a model feature because every test site is new.

In [ ]:
RAIN = ["rainfall_1h_mm", "rainfall_6h_mm", "rainfall_24h_mm", "rainfall_72h_mm"]

def build_features(df):
    df = df.sort_values(["location_id", "timestamp"]).copy()
    g = df.groupby("location_id")

    # Fill small sensor gaps with the site's previous reading (past only).
    for c in RAIN + ["soil_moisture", "vegetation_index", "temperature_c"]:
        df[c + "_ffill"] = g[c].ffill()

    doy = df.timestamp.dt.dayofyear
    df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    df["aspect_sin"], df["aspect_cos"] = np.sin(np.radians(df.aspect_deg)), np.cos(np.radians(df.aspect_deg))

    r1, r6, r24, r72 = (df[c + "_ffill"] for c in RAIN)
    df["rain_1_6h"], df["rain_6_24h"], df["rain_24_72h"] = r6 - r1, r24 - r6, r72 - r24
    df["rain_24_over_72"] = r24 / (r72 + 1.0)
    df["slope_x_moist"] = df.slope_deg * df.soil_moisture_ffill
    df["slope_x_rain72"] = df.slope_deg * r72
    df["moist_x_rain72"] = df.soil_moisture_ffill * r72
    df["sin_slope"] = np.sin(np.radians(df.slope_deg))

    # Antecedent wetness from earlier days at the same site.
    g = df.groupby("location_id")
    for c in ["rainfall_72h_mm_ffill", "soil_moisture_ffill"]:
        df[c + "_lag1"] = g[c].shift(1)
        df[c + "_lag3"] = g[c].shift(3)
        df[c + "_d1"] = df[c] - df[c + "_lag1"]
    df["rain24_sum7d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(7, min_periods=1).sum())
    df["rain24_sum30d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(30, min_periods=1).sum())
    df["moist_mean7d"] = g["soil_moisture_ffill"].transform(lambda s: s.rolling(7, min_periods=1).mean())

    df["soil_type"] = df.soil_type.astype(pd.CategoricalDtype(["sandy", "loam", "clay", "silt", "rocky", "peat"]))
    return df

full = build_features(pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True))
tr = full[full._split == "train"].reset_index(drop=True)
te = full[full._split == "test"].reset_index(drop=True)

BASE = ["elevation_m", "slope_deg", "aspect_deg", "soil_moisture", "soil_type", "vegetation_index",
        *RAIN, "temperature_c"]
ENGINEERED = BASE + ["doy_sin", "doy_cos", "aspect_sin", "aspect_cos", "rain_1_6h", "rain_6_24h",
        "rain_24_72h", "rain_24_over_72", "slope_x_moist", "slope_x_rain72", "moist_x_rain72", "sin_slope",
        "rainfall_72h_mm_ffill_lag1", "rainfall_72h_mm_ffill_lag3", "rainfall_72h_mm_ffill_d1",
        "soil_moisture_ffill_lag1", "soil_moisture_ffill_lag3", "soil_moisture_ffill_d1",
        "rain24_sum7d", "rain24_sum30d", "moist_mean7d"]

## Validation that mirrors the test set
Three folds grouped by site: each validation fold holds ~11 unseen sites, close to the 12
candidate sites per test day, and MAP@3 is computed per day among the validation-fold sites.

In [ ]:
folds = list(GroupKFold(n_splits=3).split(tr, groups=tr.location_id))

LGB_PARAMS = dict(objective="binary", learning_rate=0.02, n_estimators=2000, num_leaves=4,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                  reg_lambda=5.0, random_state=SEED, verbose=-1)

def make_lr(features):
    num = [c for c in features if c != "soil_type"]
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("soil", OneHotEncoder(handle_unknown="ignore"), ["soil_type"]),
    ])
    return make_pipeline(prep, LogisticRegression(max_iter=3000))

def evaluate_model(fit_predict):
    """Mean MAP@3 over the site-grouped folds; also returns out-of-fold scores."""
    oof, scores = np.zeros(len(tr)), []
    for fit_idx, val_idx in folds:
        oof[val_idx] = fit_predict(fit_idx, val_idx)
        scores.append(map_at_k(tr.loc[val_idx].assign(score=oof[val_idx]), "score"))
    return float(np.mean(scores)), oof

def fold_map(oof):
    return float(np.mean([map_at_k(tr.loc[v].assign(score=oof[v]), "score") for _, v in folds]))

def lr_fp(features):
    def fp(fit_idx, val_idx):
        m = make_lr(features).fit(tr.loc[fit_idx, features], tr.loc[fit_idx, TARGET])
        return m.predict_proba(tr.loc[val_idx, features])[:, 1]
    return fp

best_iters = {}
def lgb_fp(features, name):
    def fp(fit_idx, val_idx):
        m = lgb.LGBMClassifier(**LGB_PARAMS)
        m.fit(tr.loc[fit_idx, features], tr.loc[fit_idx, TARGET],
              eval_set=[(tr.loc[val_idx, features], tr.loc[val_idx, TARGET])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        best_iters.setdefault(name, []).append(m.best_iteration_)
        return m.predict_proba(tr.loc[val_idx, features])[:, 1]
    return fp

def rank_fp(features, n_estimators=300):
    """LambdaRank: optimises the within-day ordering directly; rows must be grouped by day."""
    def fp(fit_idx, val_idx):
        fit = tr.loc[fit_idx].sort_values("query_id")
        m = lgb.LGBMRanker(objective="lambdarank", learning_rate=0.03, n_estimators=n_estimators,
                           num_leaves=7, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.7, random_state=SEED, verbose=-1,
                           lambdarank_truncation_level=K)
        m.fit(fit[features], fit[TARGET], group=fit.groupby("query_id", sort=True).size().to_numpy())
        return m.predict(tr.loc[val_idx, features])
    return fp

### Step 1: simple rankers
Random order is the floor. Ranking each day's sites by 72-hour rainfall is a strong
domain heuristic that any learned model has to beat.

In [ ]:
rng = np.random.default_rng(SEED)
results, oof = {}, {}
results["random order"] = fold_map(rng.random(len(tr)))
results["heuristic: rainfall_72h"] = fold_map(tr.rainfall_72h_mm_ffill.fillna(0).to_numpy())
results["heuristic: slope x moisture x rain72"] = fold_map(
    (tr.sin_slope * tr.soil_moisture_ffill * np.log1p(tr.rainfall_72h_mm_ffill)).fillna(0).to_numpy())

### Step 2: pointwise models on raw columns, and does memorising sites help?
With site-grouped folds, `location_id` gives no gain on new sites, so it is left out.

In [ ]:
results["logistic regression, raw"], _ = evaluate_model(lr_fp(BASE))
results["LightGBM, raw"], _ = evaluate_model(lgb_fp(BASE, "raw"))
results["LightGBM, raw + location_id"], _ = evaluate_model(lgb_fp(BASE + ["location_id"], "loc"))

### Step 3: engineered features, a listwise ranker, and a blend
Rainfall increments, antecedent lags, seasonality and slope x moisture interactions are added.
LambdaRank optimises the within-day order directly. Scores from different models live on
different scales, so the blend averages within-day *ranks*.

In [ ]:
results["logistic regression, engineered"], oof["lr"] = evaluate_model(lr_fp(ENGINEERED))
results["LightGBM, engineered"], oof["lgb"] = evaluate_model(lgb_fp(ENGINEERED, "eng"))
results["LambdaRank, engineered"], oof["rank"] = evaluate_model(rank_fp(ENGINEERED))

def day_rank(scores, frame):
    return frame.assign(s=scores).groupby("query_id")["s"].rank(pct=True).to_numpy()

blend_oof = np.zeros(len(tr))
for _, v in folds:
    sub = tr.loc[v]
    blend_oof[v] = sum(day_rank(oof[m][v], sub) for m in ["lr", "lgb", "rank"]) / 3
results["blend of 3 (rank average)"] = fold_map(blend_oof)

summary = pd.Series(results, name="site-grouped MAP@3 (higher is better)").round(4)
print(summary.to_string())

## Final model
Logistic regression is the best learned ranker under site-grouped validation: the signal is
close to additive in log-odds, and the more flexible trees and LambdaRank overfit site-specific
patterns that do not transfer to new sites. The feature set (raw vs engineered) with the higher
validation MAP@3 is refit on all training sites, and each test day's top 3 sites are recommended.

In [ ]:
final_features = BASE if results["logistic regression, raw"] >= results["logistic regression, engineered"] else ENGINEERED
print("final features:", "raw" if final_features is BASE else "engineered", f"({len(final_features)} columns)")
model = make_lr(final_features).fit(tr[final_features], tr[TARGET])
te["score"] = model.predict_proba(te[final_features])[:, 1]

# Ties are broken by the smaller location_id so the output is deterministic.
top = (te.sort_values(["query_id", "score", "location_id"], ascending=[True, False, True])
         .groupby("query_id").location_id.apply(lambda s: " ".join(map(str, s.head(K)))))
submission = test_queries[["query_id"]].assign(location_ids=lambda d: d.query_id.map(top))

candidates = test_queries.set_index("query_id").candidate_sites.str.split()
assert len(submission) == len(sample) and submission.location_ids.notna().all()
assert all(len(set(r.split())) == K and set(r.split()) <= set(candidates[q])
           for q, r in zip(submission.query_id, submission.location_ids))
submission.to_csv(OUT / "submission.csv", index=False)
submission.head()